# Test: minimal canonical enumeration (`canon_minimal`)

Implements and tests the DFS + $\phi$ procedure from `writing/Canonization/2_Theory.tex`.

Same shuffle / relabel protocol as `test_canon_topo.ipynb`, but using `canon_minimal.canon_atom_num`.

In [ ]:
import random

from rdkit import Chem

from stereomolgraph import AtomId, Bond, MolGraph, StereoMolGraph
from canonical_smg import canon_atom_num, canon_phi_sequence
from stereomolgraph.ipython import View2D

View2D.show_h = True

In [ ]:
def shuffle_graph(g: StereoMolGraph | MolGraph) -> StereoMolGraph | MolGraph:
    random_atom_ids: list[AtomId] = random.sample(range(len(g)), len(g))

    old_new_mapping: dict[AtomId | None, AtomId | None] = {None: None}
    new_old_mapping: dict[AtomId | None, AtomId | None] = {None: None}

    new_g = g.__class__()

    for atom, atom_type in random.sample(list(zip(g.atoms, g.atom_types)), len(g)):
        new_atom_id = random_atom_ids.pop()
        old_new_mapping[atom] = new_atom_id
        new_old_mapping[new_atom_id] = atom
        new_g.add_atom(new_atom_id, atom_type)

    for a1, a2 in random.sample(list(g.bonds), len(g.bonds)):
        new_g.add_bond(old_new_mapping[a1], old_new_mapping[a2])

    if isinstance(g, StereoMolGraph):
        for stereo in g.atom_stereo.values():
            new_stereo = stereo.__class__(
                tuple(old_new_mapping.get(atom) for atom in stereo.atoms), stereo.parity
            )
            new_g.set_atom_stereo(new_stereo)

        for stereo in g.bond_stereo.values():
            new_stereo = stereo.__class__(
                tuple(old_new_mapping.get(atom) for atom in stereo.atoms), stereo.parity
            )
            new_g.set_bond_stereo(new_stereo)
    assert new_g == g
    return new_g

In [ ]:
from stereomolgraph.stereodescriptors import OInt


def graph_to_set(
    g: StereoMolGraph | MolGraph,
) -> set[tuple[AtomId, int] | Bond | tuple[str, int, tuple[OInt, ...]]]:
    ret: set[tuple[AtomId, int] | Bond | tuple[str, int, tuple[OInt, ...]]] = {
        (a, t) for a, t in zip(g.atoms, g.atom_types)
    }
    for bond in g.bonds:
        ret.add(bond)
    if isinstance(g, StereoMolGraph):
        for stereo in g.stereo.values():
            if None in stereo.atoms:
                stereo = stereo.__class__(
                    tuple(a if a is not None else len(g) + 1 for a in stereo.atoms),
                    stereo.parity,
                )
            ret.add((stereo.__class__.__name__, *stereo.canonical_form()))

    if isinstance(g, StereoMolGraph):
        assert len(ret) == len(g.atoms) + len(g.bonds) + len(g.stereo)
    elif isinstance(g, MolGraph):
        assert len(ret) == len(g.atoms) + len(g.bonds)

    return ret

In [ ]:
def smiles2graph(smiles: str) -> MolGraph:
    rdmol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    mg = StereoMolGraph.from_rdmol(rdmol, stereo_complete=True)
    return mg

In [ ]:
def test_canon_atom_num(g: MolGraph, n: int = 10) -> bool:
    """Shuffle atom ids n times; all canonical relabelings must coincide."""
    distinct_graphs = []
    relabeled_graphs: list[MolGraph] = []
    phi_seqs = []
    for _ in range(n):
        shuffled_g = shuffle_graph(g)
        canon_id_mapping = canon_atom_num(shuffled_g)
        # theory uses 1..|A|; relabel_atoms expects a full bijection on atom ids
        canon_graph = shuffled_g.relabel_atoms(dict(canon_id_mapping))
        graph_as_set = graph_to_set(canon_graph)
        phi_seqs.append(canon_phi_sequence(shuffled_g))
        if graph_as_set not in distinct_graphs:
            distinct_graphs.append(graph_as_set)
            relabeled_graphs.append(canon_graph)

    if len(set(phi_seqs)) != 1:
        print("Phi sequences disagree:", phi_seqs[:2])
        return False

    if len(distinct_graphs) > 1:
        display(relabeled_graphs[0])
        display(relabeled_graphs[1])
        print(distinct_graphs[0].symmetric_difference(distinct_graphs[1]))
        return False
    return True

## Smoke tests (small molecules)

In [ ]:
smoke = [
    "C",  # methane
    "C[C@H](O)N",  # chiral center
    "C[C@@H](O)N",  # enantiomer
    "C/C=C/C",  # E alkene
    "C/C=C\\C",  # Z alkene
    "c1ccccc1",  # benzene
    "CC(C)C",  # isobutane
]

for smi in smoke:
    g = smiles2graph(smi)
    ok = test_canon_atom_num(g, n=20)
    print(("PASS" if ok else "FAIL"), smi, "|A|=", len(g), "M=", dict(sorted(canon_atom_num(g).items(), key=lambda x: x[1])))

## Enantiomers should get different stereo-aware keys

Connectivity-first $\phi$ still includes stereodescriptors in the third component,
so enantiomers that share connectivity can share atom numbering on the connectivity
part but the full $\Phi$ / relabeled stereo form should differ.

In [ ]:
g_r = smiles2graph("C[C@H](O)N")
g_s = smiles2graph("C[C@@H](O)N")

set_r = graph_to_set(g_r.relabel_atoms(dict(canon_atom_num(g_r))))
set_s = graph_to_set(g_s.relabel_atoms(dict(canon_atom_num(g_s))))

print("same connectivity enumeration on atoms?", 
      {(a, t) for a, t in set_r if isinstance(a, int)} == {(a, t) for a, t in set_s if isinstance(a, int)})
print("full canonical forms equal?", set_r == set_s)
print("diff", set_r.symmetric_difference(set_s))
assert set_r != set_s, "enantiomers must not collapse under stereo-aware canon"
print("PASS enantiomer distinction")

## Difficult graphs (Krotko 2020)

These can be slow for pure DFS without color refinement pruning — start with few shuffles.

In [ ]:
difficult_graphs = [
    "C1C2CC3CC1CC(C2)C3",  # adamantane
    "C12C3C4C1C5C2C3C45",  # cubane
    "C1CC2CCC1CCC3CCC(CC3)CC2",
    "C12C3C1C4C5C4C5C23",
    # heavier examples (uncomment if patient):
    # "C12C3C4C5C1C6C7C2C8C3C6C5C8C74",  # peterson
    # "C1OC23COC45COC11COC67COC8(COC9(CO2)COC(CO1)(CO6)OCC(CO9)(OC4)OCC(CO5)(OC7)OC8)OC3",
]

for smiles in difficult_graphs:
    try:
        g = smiles2graph(smiles)
    except Exception as e:
        print(f"Error: {smiles}: {e}")
        continue
    print(f"testing {smiles} |A|={len(g)} ...", flush=True)
    ok = test_canon_atom_num(g, n=5)
    print(("PASS" if ok else "FAIL"), smiles)

## CIP Validation Suite SMILES (optional, can be slow)

Same skip list as `test_canon_topo.ipynb`.

In [ ]:
def load_smiles_from_txt(filepath: str) -> list[str]:
    with open(filepath, "r") as f:
        smiles_list = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(smiles_list)} SMILES from {filepath}")
    return smiles_list


ids_to_skip = [
    7, 22, 119, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187,  # isotopes
    10, 11,  # helicenes
    63, 78, 79, 118, 120, 135, 141, 144, 154, 164, 166, 231, 232, 243, 287,  # allenes
    23, 55, 57, 72, 73, 86, 124, 155, 158,  # atrop
]

RUN_CIP_SUITE = False  # set True to run full suite

if RUN_CIP_SUITE:
    smiles_file = load_smiles_from_txt("../data/smiles.txt")
    smiles_list = [
        smiles_file[i - 1].split()[0]
        for i, _ in enumerate(smiles_file, start=1)
        if i not in ids_to_skip
    ]
    for smiles in smiles_list:
        try:
            g = smiles2graph(smiles)
        except Exception as e:
            print(f"Error: {smiles}: {e}")
            continue
        print(smiles, end="\r", flush=True)
        if test_canon_atom_num(g, n=5):
            print(f"PASS: {smiles}")
        else:
            print(f"FAIL: {smiles}")
else:
    print("Skipped CIP suite (set RUN_CIP_SUITE = True)")